## create a connection to the database using the library sqlite3

In [13]:
import pandas as pd
import sqlite3
connection = sqlite3.connect('../../data/checking-logs.sqlite')

## using only one query for each of the groups, create two dataframes: test_results

In [ ]:
query = """SELECT d.time_flag AS time,
avg(d.mean_gap) AS avg_diff
FROM (SELECT inner_q.uid AS uid,
inner_q.time_flag,
AVG(inner_q.delta_hours) AS mean_gap
FROM (
SELECT t.uid,
CASE
WHEN t.first_commit_ts <
( SELECT MIN(DATETIME(x.first_view_ts))
FROM test x
WHERE x.uid = t.uid
AND x.labname != 'project1'
)
THEN 'before'
ELSE 'after'
END AS time_flag,
( JULIANDAY( DATETIME(dl.deadlines,'unixepoch') )
- JULIANDAY( t.first_commit_ts )
) * 24 AS delta_hours
FROM test t
right JOIN deadlines dl
ON dl.labs = t.labname
) inner_q
GROUP BY inner_q.uid,
inner_q.time_flag
) d
GROUP BY d.time_flag;
                """
test_results = pd.io.sql.read_sql(query, connection)

In [15]:
test_results

,time,avg_diff
0,after,18.116590
1,before,377.699491


In [16]:
query = """SELECT
  CASE
    WHEN strftime('%s', c.first_commit_ts) < strftime('%s', c.first_view_ts)
      THEN 'before'
    ELSE 'after'
  END AS time,
  AVG(
    (d.deadlines - strftime('%s', c.first_commit_ts)  ) / 3600
  ) AS avg_diff
FROM control c
FULL JOIN deadlines d
  ON c.labname = d.labs
WHERE c.first_view_ts IS NOT NULL
  AND d.labs != 'project1'
GROUP BY time;
                """
control_results = pd.io.sql.read_sql(query, connection)

In [17]:
control_results

,time,avg_diff
0,after,112.710526
1,before,99.464286


## close the connection

In [18]:
connection.close()